In [ ]:
%%capture
!pip install accelerate  -U # and restart the kernel

In [ ]:
%%capture
!pip install seqeval
!pip install  datasets

In [ ]:
import os, sys
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
#gpu="0,1,2,3"
gpu="0"
os.environ["CUDA_VISIBLE_DEVICES"]=gpu
import numpy as np
import seqeval.metrics
from seqeval.scheme import IOB2
import transformers
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification

from datasets import load_dataset, load_metric, Dataset, DatasetDict

from sklearn.model_selection import train_test_split

In [ ]:
MODEL = 'roberta' #or roberta
if MODEL == 'bert':
    print("Using BERT model")
    model_checkpoint = "bert-base-multilingual-cased"
    batch_size = 128
    learning_rate = 5e-5
    weight_decay = 0.0001
    epochs = 8
    warmup_steps = 3000
    seed = 1
elif MODEL == 'roberta':
    print("Using XLM-RoBERTa model")
    model_checkpoint = "xlm-roberta-base" #"xlm-roberta-large"
    batch_size = 64
    learning_rate = 1e-5
    weight_decay = 0.001
    epochs = 10
    warmup_steps = 800
    seed = 1
else:
    print("Usage example:\n python run_finetune_kaznerd.py model (bert|roberta)\n"
            "e.g.: python run_finetune_kaznerd.py roberta")
    exit()



In [ ]:
task = "ner"

#print(transformers.__version__)

def tokenize_and_align_labels(examples, tokenizer, task, label_all_tokens=False):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True,
                        is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[f"{task}_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word id that is None. We set the label to -100 so they are
            # automatically ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
                # For the other tokens in a word, we set the label to either the current label or
                # -100, depending on the label_all_tokens flag.
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [[label_list[p] for (p, l) in zip(prediction, label) if l != -100]
                            for prediction, label in zip(predictions, labels)]
    true_labels = [[label_list[l] for (p, l) in zip(prediction, label) if l != -100]
                        for prediction, label in zip(predictions, labels)]

    #computes micro average
    results = metric.compute(predictions=true_predictions, references=true_labels,
                scheme="IOB2", mode="strict")
    return {"precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"]}



In [ ]:
def read_conll2003mod(filepath: str = "something.conll2003mod-formatted.txt",
                      labels_available=True, doc_ids_available=False):
    ids, sequences, doc_ids = [], [], []

    with open(filepath, "r", encoding="utf-8") as rf:

        for line in rf:
            line = line.strip()

            # skipping empty lines
            if line:
                # reading the sentence ID
                if line.startswith("#"):
                    splitted = line.strip("#").strip().split(" ")
                    ids.append(int(splitted[0]))
                    if doc_ids_available:
                        doc_ids.append(int(splitted[1]))
                    sequences.append([])

                # reading the token-tag pair
                else:
                    token = line.split("\t")[0]
                    if labels_available:
                        tag = line.split("\t")[3]
                    else:
                        tag = -1
                    sequences[-1].append((token, tag))

    assert len(ids) == len(sequences)

    if doc_ids_available:
        return ids, sequences, doc_ids

    return ids, sequences

# def read_conll_file(file_path):
#     with open(file_path, "r") as f:
#         content = f.read().strip()
#         sentences = content.split("\n\n")
#         data = []
#         for sentence in sentences:
#             tokens = sentence.split("\n")
#             token_data = []
#             for token in tokens:
#                 token_data.append(token.split())
#             data.append(token_data)
#     return data


# train_data = read_conll_file("/content/drive/MyDrive/The_Cramer/AkylAI/NER/Codes/KyrgyzNER_TRAIN.jsonl.txt")
# validation_data = read_conll_file("/kaggle/input/conll2003-dataset/conll2003/eng.testa")
# test_data = read_conll_file("/kaggle/input/conll2003-dataset/conll2003/eng.testb")

data_id, data = read_conll2003mod('KyrgyzNER_TRAIN.jsonl.txt')

test_id, test = read_conll2003mod('KyrgyzNER_TEST.jsonl.gold.txt')

train, valid = train_test_split(data, random_state=42, test_size=0.2)

print(len(train), len(valid), len(test))

label_list_train = set([token_data[1] for sentence in data for token_data in sentence])
label_list_test = set([token_data[1] for sentence in test for token_data in sentence])
label_list = list(label_list_train | label_list_test)

label_map = {label: i for i, label in enumerate(label_list)}

print(len(label_list))


In [ ]:
label_list

In [ ]:
label_map

In [ ]:
def convert_to_dataset(data):
    formatted_data = {"tokens": [], "ner_tags": []}

    for sentence in data:

        tokens = [token_data[0] for token_data in sentence]
        ner_tags = [label_map[token_data[1]]  for token_data in sentence]
        # ner_tags = [token_data[1]  for token_data in sentence]

        formatted_data["tokens"].append(tokens)
        formatted_data["ner_tags"].append(ner_tags)

    return Dataset.from_dict(formatted_data)


train_dataset = convert_to_dataset(train)
validation_dataset = convert_to_dataset(valid)
test_dataset = convert_to_dataset(test)


datasets = DatasetDict({
    "train": train_dataset,
    "validation": validation_dataset,
    "test": test_dataset,
})


In [ ]:

# datasets = load_dataset("kaznerd.py")
# label_list = datasets["train"].features[f"{task}_tags"].feature.names


tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

assert isinstance(tokenizer, transformers.PreTrainedTokenizerFast)
data_collator = DataCollatorForTokenClassification(tokenizer)

metric = load_metric("seqeval")

model = AutoModelForTokenClassification.from_pretrained(model_checkpoint,
            num_labels=len(label_list))

model_name = model_checkpoint.split("/")[-1]

args = TrainingArguments(f"{model_name}-kyrgyzNER",
                            overwrite_output_dir=True,
                            evaluation_strategy="epoch",
                            per_device_train_batch_size=batch_size,
                            per_device_eval_batch_size=batch_size,
                            learning_rate=learning_rate,
                            num_train_epochs=epochs,
                            warmup_steps=warmup_steps,
                            weight_decay=weight_decay,
                            save_strategy="no",
                            seed=seed,
                            push_to_hub=False)

tokenized_datasets = datasets.map(tokenize_and_align_labels, batched=True,
        fn_kwargs={"tokenizer":tokenizer,"task":task})

trainer = Trainer(model, args,
                  data_collator=data_collator,
                  train_dataset=tokenized_datasets["train"],
                  eval_dataset=tokenized_datasets["validation"],
                  tokenizer=tokenizer,
                  compute_metrics=compute_metrics)

trainer.train()
#trainer.evaluate()


In [ ]:

#################################################################################################
#Evaluate validation set
print("#"*100)
predictions, labels, _ = trainer.predict(tokenized_datasets["validation"])
predictions = np.argmax(predictions, axis=2)

# Remove ignored index (special tokens)
true_predictions = [[label_list[p] for (p, l) in zip(prediction, label) if l != -100]
                    for prediction, label in zip(predictions, labels)]
true_labels = [[label_list[l] for (p, l) in zip(prediction, label) if l != -100]
               for prediction, label in zip(predictions, labels)]

results = metric.compute(predictions=true_predictions, references=true_labels, scheme="IOB2",
            mode="strict")
print("\nValidation: Overall F1", results["overall_f1"])
print("Validation: Total number of sentences:",len(true_labels))
print("Validation: Total number of tokens:", sum([len(sent) for sent in true_labels]))
print("Validation: seqeval based results")
print(seqeval.metrics.classification_report(true_labels, true_predictions, digits=4, mode='strict',
    scheme=IOB2))


In [ ]:

#################################################################################################
#Evaluate test set
print("#"*100)
predictions, labels, _ = trainer.predict(tokenized_datasets["test"])
predictions = np.argmax(predictions, axis=2)

# Remove ignored index (special tokens)
true_predictions = [[label_list[p] for (p, l) in zip(prediction, label) if l != -100]
                    for prediction, label in zip(predictions, labels)]
true_labels = [[label_list[l] for (p, l) in zip(prediction, label) if l != -100]
               for prediction, label in zip(predictions, labels)]

results = metric.compute(predictions=true_predictions, references=true_labels, scheme="IOB2",
            mode="strict")
print("\nTest: Overall F1", results["overall_f1"])
print("Test: Total number of sentences:",len(true_labels))
print("Test: Total number of tokens:", sum([len(sent) for sent in true_labels]))
print("Test: seqeval based results")
print(seqeval.metrics.classification_report(true_labels, true_predictions, digits=4, mode='strict',
    scheme=IOB2))
print("#"*100)

In [ ]:

label_map_id2label = {j:i for i, j in label_map.items() }
label_map_id2label

In [ ]:
# model.config.id2label UPDATE: Done

model.config.id2label.update(label_map_id2label)
model.config.id2label

In [ ]:
model.config.label2id = {j:i for i, j in model.config.id2label.items()}
model.config.label2id